In [ ]:
import re
import pandas as pd
from collections import Counter

df=pd.read_csv(r"C:/Users/yourname/Documents/DATASET.csv")

def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # remove anything in brackets (round, square, curly)
    text = re.sub(r'\(.*?\)|\[.*?\]|\{.*?\}', '', text)
    # remove "limited" or "ltd" (case insensitive, as whole words)
    text = re.sub(r'\bltd\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\blimited\b', '', text, flags=re.IGNORECASE)
    # remove all whitespace
    text = re.sub(r'\s+', '', text)
    return text.strip().lower()

def size_category(value):
    if pd.isna(value):
        return None
    if value < 51:
        return 'Small'
    elif value <= 200:
        return 'Medium'
    else:  # >= 201
        return 'Large'

# filter to relevant years
df_filtered = df[(df['Year'] >= 2015) & (df['Year'] <= 2024)].copy()

# apply normalization and size bucketing
df_filtered['B_norm'] = df_filtered['Company Name'].apply(normalize)
df_filtered['Size'] = df_filtered['NO. OF GUARDS EMPLOYED'].apply(size_category)  # <-- using column E

# drop rows where size couldn't be determined
df_filtered = df_filtered.dropna(subset=['Size'])

# build, for each year, a mapping of normalized name -> Size
# (if duplicates exist per year, this keeps the LAST occurrence — see note below)
year_maps = (
    df_filtered.groupby('Year')
    .apply(lambda g: dict(zip(g['B_norm'], g['Size'])))
    .to_dict()
)

years = sorted(year_maps.keys())

transition_labels = ['Small-Medium', 'Small-Large', 'Medium-Small', 'Medium-Large', 'Large-Small', 'Large-Medium']
results = []

for y1, y2 in zip(years, years[1:]):
    map1 = year_maps[y1]
    map2 = year_maps[y2]
    common_names = set(map1.keys()) & set(map2.keys())

    counts = Counter()
    for name in common_names:
        s1, s2 = map1[name], map2[name]
        if s1 != s2:
            counts[f"{s1}-{s2}"] += 1

    row = {'Year_Pair': f"{y1}-{y2}"}
    for label in transition_labels:
        row[label] = counts.get(label, 0)
    results.append(row)

results_df = pd.DataFrame(results, columns=['Year_Pair'] + transition_labels)
print(results_df)

   Year_Pair  Small-Medium  Small-Large  Medium-Small  Medium-Large  \
0  2015-2016             4            0             2             1   
1  2016-2017             6            2             1             2   
2  2017-2018             5            0             5             1   
3  2018-2019             3            0             0             2   
4  2019-2020             4            0             4             1   
5  2020-2021             5            0             3             1   
6  2021-2022             6            0             3             1   
7  2022-2023             5            0             3             2   
8  2023-2024             7            0             3             1   

   Large-Small  Large-Medium  
0            0             1  
1            0             1  
2            1             2  
3            0             0  
4            0             0  
5            0             0  
6            0             1  
7            0             2  
8         

C:\Users\bradj\AppData\Local\Temp\ipykernel_25244\1762333623.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: dict(zip(g['B_norm'], g['Size'])))


In [ ]:
# This was for additional testing and checking. Not really used in the analysis but I thought I would leave it in anyway for interest. 

transition_records = []

for y1, y2 in zip(years, years[1:]):
    map1 = year_maps[y1]
    map2 = year_maps[y2]

    df1 = df_filtered[df_filtered['Year'] == y1]
    df2 = df_filtered[df_filtered['Year'] == y2]

    # normalized -> original name lookup
    orig1 = dict(zip(df1['B_norm'], df1['Company Name']))
    orig2 = dict(zip(df2['B_norm'], df2['Company Name']))

    common_names = set(map1.keys()) & set(map2.keys())

    for name in common_names:
        s1, s2 = map1[name], map2[name]
        if s1 != s2:
            transition_records.append({
                'Year_Pair': f"{y1}-{y2}",
                'Transition': f"{s1}-{s2}",
                'Company_Name': orig1.get(name, None)
            })

company_results_df = pd.DataFrame(transition_records)

# Group them so each transition has a full list of names
grouped = (
    company_results_df
    .groupby(['Year_Pair', 'Transition'])['Company_Name']
    .apply(list)
    .reset_index()
)

# PRINT WITHOUT TRUNCATION
print(grouped.to_string(max_rows=None, max_cols=None))

    Year_Pair    Transition                                                                                                                                                                Company_Name
0   2015-2016  Large-Medium                                                                                                                                            [WAP & CO LTD SECURITY SERVICES]
1   2015-2016  Medium-Large                                                                                                                                                   [WASMAN SECURITY LIMITED]
2   2015-2016  Medium-Small                                                                                                                     [ASSET PROTECTION DEPARTMENT, EVIEVI SECURITY SERVICES]
3   2015-2016  Small-Medium                                                                            [PAC SECURITY SERVICES, QPR ESCORT LIMITED, ALIR SECURITY SERVICES, KOLI SECURITY SOLUTIONS LTD]


In [6]:
# --- COMPANY TRANSITIONS WITH LICENSE CLASS IN BOTH YEARS ---

transition_records = []

for y1, y2 in zip(years, years[1:]):
    map1 = year_maps[y1]
    map2 = year_maps[y2]

    df1 = df_filtered[df_filtered['Year'] == y1]
    df2 = df_filtered[df_filtered['Year'] == y2]

    # normalized -> original name lookup
    orig1 = dict(zip(df1['B_norm'], df1['Company Name']))
    orig2 = dict(zip(df2['B_norm'], df2['Company Name']))

    # normalized -> license class lookup
    lic1 = dict(zip(df1['B_norm'], df1['License Class']))
    lic2 = dict(zip(df2['B_norm'], df2['License Class']))

    common_names = set(map1.keys()) & set(map2.keys())

    for name in common_names:
        s1, s2 = map1[name], map2[name]
        if s1 != s2:
            transition_records.append({
                'Year_Pair': f"{y1}-{y2}",
                'Transition': f"{s1}-{s2}",
                'Company_Name': orig1.get(name),
                'License_Y1': lic1.get(name),
                'License_Y2': lic2.get(name)
            })

company_results_df = pd.DataFrame(transition_records)

# Group so each transition has full lists of companies + license classes
grouped = (
    company_results_df
    .groupby(['Year_Pair', 'Transition'])[['Company_Name', 'License_Y1', 'License_Y2']]
    .apply(lambda x: x.to_dict('records'))
    .reset_index()
)

# Print without truncation
print(grouped.to_string(max_rows=None, max_cols=None))

    Year_Pair    Transition                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 0
0   2015-2016  Large-Medium                                                                                                                                                                                                                                                                                                                                                                               